# Copying FASTQs from Illumina BaseSpace to APGAP

The data-ingress notebook. Sits at the front of the APGAP workflow: gets raw sequencing data from Illumina BaseSpace into the platform where it can feed [02-read-your-data.ipynb](02-read-your-data.ipynb), [03-launch-a-pipeline.ipynb](03-launch-a-pipeline.ipynb), and downstream analysis.

**How this works (short version):**

1. In the APGAP portal, you create a **batch upload endpoint** (BaseSpace kind). The Portal generates a service account key JSON and an ingest bucket URL.
2. You paste both the URL and the SA key JSON into a small widget that appears near the top of this notebook. The notebook saves them to files in `~/.apgap/` with restricted permissions and reuses them on subsequent runs.
3. The notebook downloads selected files from BaseSpace to a Workbench temp directory, then uploads them to the ingest bucket using the SA key.
4. Once files land in the ingest bucket, the APGAP backend takes over automatically: DLP scan → SRA scrubber (human read removal) → move to your lab's target bucket. Files then show up in [02-read-your-data.ipynb](02-read-your-data.ipynb).

You don't touch anything downstream of the ingest bucket write. The notebook is done as soon as its uploads finish.

**What's happening under the hood** (BaseSpace concepts, DLP scanning, the scrubber cascade, batch upload endpoints) is explained in [05-reference.ipynb](05-reference.ipynb). This notebook stays focused on running the transfer.

**Time and cost:**

- **One-time setup**: ~2 min for `bs auth` (Illumina login), ~1 min to create the Portal endpoint
- **Transfer**: dominated by network. Typical MiSeq run (~50 samples, ~10 GB total) finishes in 10-20 min on a Workbench VM
- **Cost**: cents for storage; no GCP Batch, no VM allocation, no Nextflow overhead


> **Kernel:** When JupyterLab prompts "Select Kernel," pick **`Python 3 (Local)`** (under "Start python Kernel"). Don't pick `Python 3 (ipykernel) (Local)`, PyTorch, or TensorFlow — those are missing libraries this notebook needs and will fail at the first import call with `ModuleNotFoundError`. If the top-right of this tab already shows `Python 3 (Local)`, you're good.


## Tools you'll use

- **Illumina BaseSpace CLI (`bs`)** — auth to Illumina, list projects, download files
- **google-cloud-storage** (Python) — upload files to the ingest bucket using the endpoint's service account key
- **APGAP Portal** — creating the batch upload endpoint (browser flow, one-time per transfer session)

Unlike notebooks 03 and 06, no Nextflow or GCP Batch here. This is a straight Python transfer script.


In [ ]:
# --- BaseSpace source ---
# BaseSpace project ID. Leave blank to list your projects interactively
# (or auto-select if you only have access to one). The choice gets saved
# to ~/.apgap/endpoint.json for future runs.
BASESPACE_PROJECT_ID = ""                                  # @param {type:"string"}

# Which files to transfer.
#   True  = only FASTQ files (.fastq.gz, .fastq, .fq.gz) — recommended default
#           for feeding viralrecon / tostadas. Skips BaseSpace's workflow
#           metadata (log files, .command.*, workflow.db, etc.) which users
#           rarely need for downstream analysis.
#   False = transfer everything visible in the project.
FASTQ_ONLY = True                                          # @param {type:"boolean"}

# Alternatively, hand-pick specific BaseSpace file IDs. If non-empty, this
# overrides FASTQ_ONLY (transfers exactly this list). Leave empty to use
# the FASTQ_ONLY filter above.
SELECTED_FILE_IDS = []                                     # @param

# Force re-transfer of files that appear in ~/.apgap/transferred.json.
#   False = skip files we've already transferred (safe default; avoids
#           creating duplicates in the lab bucket when the ingest bucket
#           is empty because the scrubber cascade already processed them)
#   True  = ignore the transfer log and re-download + re-upload everything
#           that matches FASTQ_ONLY / SELECTED_FILE_IDS. Use this when
#           you've deleted files from the lab and want to re-transfer.
# For surgical re-upload of specific files, combine with SELECTED_FILE_IDS.
FORCE_REUPLOAD = False                                     # @param {type:"boolean"}

# --- Local staging ---
# Where downloads from BaseSpace get buffered on the Workbench VM before
# upload. Cleaned up after each file. Default should be fine unless your
# home dir is unusually small.
LOCAL_STAGING_DIR = "/tmp/basespace-transfer"              # @param {type:"string"}

# --- Advanced: where the endpoint config gets saved ---
# The widget in the next cell writes the SA key + ingest bucket URL to
# these two files. Defaults are a per-user location outside any git repo.
# Change only if you have a good reason (e.g., multiple endpoints on the
# same VM). Both files live in the same directory.
SERVICE_ACCOUNT_KEY_PATH = "~/.apgap/sa-key.json"          # @param {type:"string"}

# Endpoint config path (URL + metadata) is derived from the SA key path so
# both files always live together in the same directory.
import os
SERVICE_ACCOUNT_KEY_PATH = os.path.expanduser(SERVICE_ACCOUNT_KEY_PATH)
ENDPOINT_CONFIG_PATH = os.path.join(
    os.path.dirname(SERVICE_ACCOUNT_KEY_PATH), "endpoint.json"
)


## Paste the endpoint URL and service key

The APGAP Portal gives you two things when you create a batch upload endpoint: the ingest bucket URI (just the bucket name), and the service key JSON (copy-to-clipboard). Paste both into the widget below and click **Save**. The notebook writes them to:

- `~/.apgap/sa-key.json` — the SA key JSON, `chmod 600` (owner-only readable)
- `~/.apgap/endpoint.json` — the ingest bucket URL and any related metadata

**Two safety features:**

- Widget contents are runtime-only, not saved into the `.ipynb` file — so the private key can't leak into a git commit
- Both files live outside any git repo by default

If both files already exist from a previous run, this cell skips the widget and reuses the saved values. To re-paste (e.g., for a new endpoint), delete `~/.apgap/sa-key.json` and `~/.apgap/endpoint.json`.


In [ ]:
import json
import os

os.makedirs(os.path.dirname(SERVICE_ACCOUNT_KEY_PATH), exist_ok=True)

_both_exist = (
    os.path.exists(SERVICE_ACCOUNT_KEY_PATH)
    and os.path.exists(ENDPOINT_CONFIG_PATH)
)

if _both_exist:
    print(f"Endpoint config already saved:")
    print(f"  SA key:   {SERVICE_ACCOUNT_KEY_PATH}")
    print(f"  Config:   {ENDPOINT_CONFIG_PATH}")
    print("Skipping paste widget. Delete both files to re-paste for a new endpoint.")
else:
    import ipywidgets as widgets
    from IPython.display import display

    print("Paste both fields from the Portal endpoint, then click Save.")

    # Pre-fill URL field if a partial config from a prior partial run exists.
    _prefill_url = ""
    if os.path.exists(ENDPOINT_CONFIG_PATH):
        try:
            _prefill_url = json.load(open(ENDPOINT_CONFIG_PATH)).get("ingest_bucket_url", "")
        except Exception:
            pass

    url_widget = widgets.Text(
        value=_prefill_url,
        placeholder="Paste the ingest bucket URI (prepend gs:// if the Portal only shows the bucket name)",
        description="Bucket URL:",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "initial"},
    )
    key_widget = widgets.Textarea(
        placeholder='Paste the entire SA key JSON here (starts with { "type": "service_account", ...)',
        description="SA key JSON:",
        layout=widgets.Layout(width="100%", height="220px"),
        style={"description_width": "initial"},
    )
    save_button = widgets.Button(
        description="Save endpoint config",
        button_style="primary",
    )
    output = widgets.Output()

    def _on_save(_):
        with output:
            output.clear_output()

            # Validate URL
            url = url_widget.value.strip()
            if not url:
                print("ERROR: bucket URL is empty.")
                return
            if not url.startswith("gs://"):
                # Portal displays the URI without the gs:// scheme; fix it up
                # rather than erroring, since this is a common paste mistake.
                url = "gs://" + url
                print(f"NOTE: prepended gs:// to URL (Portal shows just the bucket name).")
            if not url.startswith("gs://batch-upload-"):
                print(
                    f"WARNING: URL doesn't match expected pattern "
                    f"(gs://batch-upload-lab<N>-<ts>-<uuid>). Got: {url}\n"
                    f"  Continuing anyway, but double-check you copied from the right endpoint."
                )

            # Validate SA key JSON
            content = key_widget.value.strip()
            if not content:
                print("ERROR: SA key JSON is empty.")
                return
            try:
                parsed = json.loads(content)
            except json.JSONDecodeError as e:
                print(f"ERROR: SA key is not valid JSON: {e}")
                return
            if parsed.get("type") != "service_account" or "private_key" not in parsed:
                print(
                    "ERROR: this doesn't look like a GCP service account key. "
                    "Expected type=service_account and a private_key field. "
                    "Re-copy from the Portal."
                )
                return

            # Write both files
            with open(SERVICE_ACCOUNT_KEY_PATH, "w") as f:
                f.write(content)
            os.chmod(SERVICE_ACCOUNT_KEY_PATH, 0o600)
            with open(ENDPOINT_CONFIG_PATH, "w") as f:
                json.dump({"ingest_bucket_url": url}, f, indent=2)
            os.chmod(ENDPOINT_CONFIG_PATH, 0o600)

            print(f"Saved SA key to {SERVICE_ACCOUNT_KEY_PATH}")
            print(f"Saved endpoint config to {ENDPOINT_CONFIG_PATH}")
            print(f"  ingest_bucket_url: {url}")
            print(f"  client_email:      {parsed.get('client_email', '?')}")
            print("\nContinue running the rest of the notebook.")

    save_button.on_click(_on_save)
    display(widgets.VBox([url_widget, key_widget, save_button, output]))


In [ ]:
import json
import os
import subprocess

# Verify the SA key file exists and looks like a GCP service account JSON.
# On first run the widget cell above creates this file; on subsequent runs
# it's reused. If we get here without the file, the user hasn't clicked
# Save in the widget yet.
if not os.path.exists(SERVICE_ACCOUNT_KEY_PATH):
    raise FileNotFoundError(
        f"SERVICE_ACCOUNT_KEY_PATH does not exist: {SERVICE_ACCOUNT_KEY_PATH}\n"
        f"Go back to the endpoint widget cell above, paste the SA key JSON and "
        f"bucket URL from the Portal, click Save, then re-run from this cell "
        f"forward."
    )

with open(SERVICE_ACCOUNT_KEY_PATH) as f:
    sa_key = json.load(f)

if sa_key.get("type") != "service_account" or "private_key" not in sa_key:
    raise ValueError(
        f"{SERVICE_ACCOUNT_KEY_PATH} does not look like a GCP service account "
        f"key. Expected type=service_account and a private_key field. Delete "
        f"{SERVICE_ACCOUNT_KEY_PATH} and {ENDPOINT_CONFIG_PATH} and re-paste "
        f"via the widget above."
    )

# Load ingest bucket URL from the endpoint config file (also written by
# the widget cell). Fail with a clear message if the widget hasn't been
# run yet.
if not os.path.exists(ENDPOINT_CONFIG_PATH):
    raise FileNotFoundError(
        f"ENDPOINT_CONFIG_PATH does not exist: {ENDPOINT_CONFIG_PATH}\n"
        f"Go back to the endpoint widget cell above and click Save."
    )
_endpoint_config = json.load(open(ENDPOINT_CONFIG_PATH))
INGEST_BUCKET_URL = _endpoint_config.get("ingest_bucket_url", "")

print(f"Service account: {sa_key.get('client_email', '(unknown)')}")
print(f"Project:         {sa_key.get('project_id', '(unknown)')}")
print(f"Ingest bucket:   {INGEST_BUCKET_URL}")
print()

if not INGEST_BUCKET_URL:
    raise ValueError(
        f"{ENDPOINT_CONFIG_PATH} has no ingest_bucket_url. Delete both files "
        f"({SERVICE_ACCOUNT_KEY_PATH} and {ENDPOINT_CONFIG_PATH}) and re-paste "
        f"via the widget above."
    )
if not INGEST_BUCKET_URL.startswith("gs://batch-upload-"):
    print(
        f"WARNING: INGEST_BUCKET_URL doesn't match the expected pattern "
        f"(gs://batch-upload-lab<N>-<ts>-<uuid>). Got: {INGEST_BUCKET_URL}\n"
        f"  Continuing anyway, but double-check you copied the right URL "
        f"from the Portal endpoint."
    )

# Create staging dir.
os.makedirs(LOCAL_STAGING_DIR, exist_ok=True)
print(f"Local staging:   {LOCAL_STAGING_DIR}")

# Best-effort git safety warning: fires only if the user set
# SERVICE_ACCOUNT_KEY_PATH to somewhere inside a git repo that isn't
# gitignored. The default path (~/.apgap/sa-key.json) is outside any
# repo so this is a no-op for most users; kept as a guard for anyone
# who overrides the default.
sa_dir = os.path.dirname(os.path.abspath(SERVICE_ACCOUNT_KEY_PATH)) or "."
sa_basename = os.path.basename(SERVICE_ACCOUNT_KEY_PATH)
in_repo = subprocess.run(
    ["git", "-C", sa_dir, "rev-parse", "--is-inside-work-tree"],
    capture_output=True, text=True,
).returncode == 0
if in_repo:
    ignored = subprocess.run(
        ["git", "-C", sa_dir, "check-ignore", "-q", sa_basename],
        capture_output=True,
    ).returncode == 0
    if not ignored:
        print(
            f"\nWARNING: {SERVICE_ACCOUNT_KEY_PATH} lives in a git repo and "
            f"is NOT gitignored. Move the file outside the repo or add its "
            f"filename to .gitignore before committing anything from this "
            f"directory."
        )


## Installing the BaseSpace CLI and google-cloud-storage

The Workbench image doesn't ship with either of these. The BaseSpace CLI (`bs`) is a standalone binary from Illumina (not on conda), so we download it directly to `~/.local/bin`. The Python `google-cloud-storage` client goes in via `pip install --user`.

The cell is idempotent — no-ops if both are already installed.


In [ ]:
import os
import stat
import subprocess
import sys
from urllib.request import urlopen, HTTPError

# --- BaseSpace CLI (bs binary from Illumina) ---
# Pinned to a specific version. Using "latest" is convenient but fragile:
# if Illumina changes flag names between releases (as they did with the
# --limit flag removal), notebooks silently break. Bump BS_VERSION when
# validating against a newer bs release.
BS_VERSION = "1.7.0"
BS_INSTALL_DIR = os.path.expanduser("~/.local/bin")
BS_BINARY = os.path.join(BS_INSTALL_DIR, "bs")
BS_URL = f"https://launch.basespace.illumina.com/CLI/{BS_VERSION}/amd64-linux/bs"

if os.path.exists(BS_BINARY):
    print(f"BaseSpace CLI already present at {BS_BINARY}, skipping download.")
else:
    os.makedirs(BS_INSTALL_DIR, exist_ok=True)
    print(f"Downloading BaseSpace CLI {BS_VERSION} from Illumina to {BS_BINARY}...")
    try:
        with urlopen(BS_URL) as resp, open(BS_BINARY, "wb") as out:
            out.write(resp.read())
    except HTTPError as e:
        raise RuntimeError(
            f"Failed to download bs from {BS_URL} (HTTP {e.code}). "
            f"Illumina may have removed version {BS_VERSION}. Bump BS_VERSION "
            f"to a newer release (see https://developer.basespace.illumina.com/"
            f"docs/content/documentation/cli/cli-overview for current version) "
            f"and re-run this cell."
        )
    st = os.stat(BS_BINARY)
    os.chmod(BS_BINARY, st.st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
    print(f"Installed BaseSpace CLI: {BS_BINARY}")

# Make sure ~/.local/bin is on PATH for subsequent subprocess calls.
if BS_INSTALL_DIR not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] = BS_INSTALL_DIR + os.pathsep + os.environ["PATH"]

# Print the actual version and warn if it doesn't match what we pinned.
# Mismatch would mean the download succeeded but returned a different
# version than expected -- worth flagging so troubleshooting can start
# from the right place.
print()
version_result = subprocess.run(["bs", "--version"], capture_output=True, text=True)
print(version_result.stdout.strip())
if BS_VERSION not in version_result.stdout:
    print(f"\nWARNING: installed bs version doesn't contain the expected pinned "
          f"string '{BS_VERSION}'. Notebook was validated against {BS_VERSION}; "
          f"if commands start failing with unknown-flag errors, that's why.")

# --- google-cloud-storage Python client ---
try:
    import google.cloud.storage  # noqa: F401
    print(f"\ngoogle-cloud-storage already importable, skipping pip install.")
except ImportError:
    print("\nInstalling google-cloud-storage via pip --user...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--user", "--quiet", "google-cloud-storage"],
        check=True,
    )
    # Make sure the user site-packages is on sys.path for this kernel session
    # (same pattern as nb 02 and nb 04 for pip --user installs).
    import site
    user_site = site.getusersitepackages()
    if user_site not in sys.path:
        sys.path.insert(0, user_site)
        print(f"Added {user_site} to sys.path")

import google.cloud.storage
print(f"google-cloud-storage version: {google.cloud.storage.__version__}")
print("\nInstall complete.")


## Authenticating with BaseSpace

Illumina requires per-user browser-based authentication (OAuth, same pattern as `gcloud auth login`). This is a **one-time setup per Workbench VM** — credentials cache to `~/.basespace/default.cfg` and persist across sessions, so future runs skip this step entirely.

**How to do it** (first-time setup only):

1. **Open a JupyterLab terminal.** Click the **+** button in the file browser (top-left), then click **Terminal** under "Other". A terminal tab opens.
2. **Run the auth command** in that terminal:
   ```
   ~/.local/bin/bs auth
   ```
   The full path is needed because the Workbench shell doesn't auto-load `~/.local/bin` where the install cell dropped the `bs` binary.
3. The command will print something like:
   ```
   Please go to this URL to authenticate:
   https://basespace.illumina.com/oauth/device?user_code=XXXX-XXXX
   ```
4. **Copy the URL** and paste it into any browser tab (doesn't have to be on the Workbench). Log in with your Illumina account, click **Accept** on the authorization screen.
5. Back in the terminal, `bs auth` detects the completion and prints something like:
   ```
   You are now authorized as user@example.com
   Access token successfully saved to /home/jupyter/.basespace/default.cfg
   ```
6. **Come back to this notebook tab and re-run the cell below.** It'll now find the cached credentials and print a `bs whoami` sanity check.


In [ ]:
import os
import subprocess

BS_CONFIG = os.path.expanduser("~/.basespace/default.cfg")

if os.path.exists(BS_CONFIG):
    print(f"BaseSpace credentials found at {BS_CONFIG}")
    result = subprocess.run(
        ["bs", "whoami"],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        print(result.stdout.strip())
        print("\nAuth is valid, moving on.")
    else:
        print("Cached credentials failed the `bs whoami` check:")
        print(result.stderr.strip())
        print()
        print("Re-authenticate: in a JupyterLab terminal, run `~/.local/bin/bs auth`")
        print("and complete the browser flow. See the markdown cell above for step-by-step.")
        raise RuntimeError("BaseSpace credentials expired or invalid. Re-run `~/.local/bin/bs auth`.")
else:
    print("No BaseSpace credentials found. First-time setup needed.")
    print()
    print("STEP 1: Open a JupyterLab terminal (+ button > Terminal under 'Other')")
    print()
    print("STEP 2: In the terminal, run:")
    print()
    print("    ~/.local/bin/bs auth")
    print()
    print("STEP 3: Copy the URL it prints, paste into a browser, log in with Illumina, click Accept.")
    print()
    print("STEP 4: Wait for the terminal to confirm 'Access token successfully saved'.")
    print()
    print("STEP 5: Come back here and re-run this cell.")
    print()
    print("See the markdown cell above for the full walkthrough with what to expect at each step.")
    raise RuntimeError("BaseSpace auth required. Run `~/.local/bin/bs auth` in a terminal (see steps above).")


## Finding your files on BaseSpace

Project selection is automatic in the common case:

- **You have access to one BaseSpace project** → auto-selected, no user action needed
- **You have access to multiple projects** → a dropdown appears; pick one and click Use this project
- **You've already picked in a previous run** → the choice is saved to `~/.apgap/endpoint.json` and reused
- **You want to override any of the above** → set `BASESPACE_PROJECT_ID` in the parameter cell (highest priority)

Once the project is resolved, the cell lists all files in it. By default, this notebook will copy **every** file in the project — set `SELECTED_FILE_IDS` in the parameter cell if you only want a subset (numeric BaseSpace file IDs as strings).


In [ ]:
import json
import subprocess

# The Illumina bs CLI doesn't expose a --limit flag on its list subcommands
# (checked with `bs project list --help` on version 1.7.0). It returns
# everything in one call. If a future bs version changes that and starts
# truncating, we'll need to add pagination via --offset here.

# File extensions that count as "FASTQ" data files. Used to filter out
# BaseSpace's workflow metadata files (log/config/db) that users don't
# need for downstream analysis. Matched case-insensitively.
FASTQ_EXTENSIONS = (".fastq.gz", ".fastq", ".fq.gz", ".fq")


def _run_bs(args):
    """Run a bs command, print stderr on failure so the user sees the actual
    error instead of a generic CalledProcessError with the stderr hidden.
    """
    result = subprocess.run(args, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"bs command failed with exit code {result.returncode}:")
        print(f"  command: {' '.join(args)}")
        print(f"  stderr:")
        for line in (result.stderr or "").splitlines():
            print(f"    {line}")
        if result.stdout:
            print(f"  stdout:")
            for line in result.stdout.splitlines():
                print(f"    {line}")
        raise RuntimeError(
            f"bs {args[1]} {args[2] if len(args) > 2 else ''} failed. "
            f"See stderr above."
        )
    return result.stdout


def _save_selection_to_endpoint(project_id, project_name):
    """Persist the chosen BaseSpace project into endpoint.json so future
    notebook runs skip the picker."""
    config = json.load(open(ENDPOINT_CONFIG_PATH))
    config["basespace_project_id"] = str(project_id)
    config["basespace_project_name"] = project_name
    with open(ENDPOINT_CONFIG_PATH, "w") as f:
        json.dump(config, f, indent=2)


def _is_fastq(filename):
    return filename.lower().endswith(FASTQ_EXTENSIONS)


# Resolve BASESPACE_PROJECT_ID in priority order:
#   1. User explicitly set it in the parameter cell (highest priority)
#   2. It's saved in endpoint.json from a previous run
#   3. Only one project is visible to the user -> auto-select
#   4. Multiple projects visible -> show dropdown widget
if BASESPACE_PROJECT_ID:
    print(f"Using BASESPACE_PROJECT_ID from parameter cell: {BASESPACE_PROJECT_ID}")
else:
    _saved = json.load(open(ENDPOINT_CONFIG_PATH)).get("basespace_project_id", "")
    if _saved:
        BASESPACE_PROJECT_ID = _saved
        _saved_name = json.load(open(ENDPOINT_CONFIG_PATH)).get("basespace_project_name", "")
        print(f"Using project saved in endpoint.json: {_saved_name} (Id={BASESPACE_PROJECT_ID})")
        print("  (delete ~/.apgap/endpoint.json to pick a different project)")
    else:
        # Fetch the list of projects the user has access to.
        projects = json.loads(_run_bs(["bs", "project", "list", "-f", "json"]))
        if not projects:
            raise RuntimeError(
                "Your Illumina account has no visible BaseSpace projects. "
                "Ask whoever shared the sequencing data with you to confirm "
                "the invitation went to smanda@asu.edu (or whichever account "
                "you `bs auth`'d with)."
            )

        if len(projects) == 1:
            # Auto-select the only project. Save to endpoint.json so future
            # runs skip the discovery step entirely.
            p = projects[0]
            BASESPACE_PROJECT_ID = str(p["Id"])
            _save_selection_to_endpoint(BASESPACE_PROJECT_ID, p.get("Name", ""))
            print(f"Auto-selected the only project you have access to:")
            print(f"  {p.get('Name', '?')} (Id={BASESPACE_PROJECT_ID})")
            print(f"  Saved to {ENDPOINT_CONFIG_PATH}")
        else:
            # Multiple projects -> show dropdown widget.
            import ipywidgets as widgets
            from IPython.display import display

            print(f"You have access to {len(projects)} BaseSpace projects. Pick one:\n")
            dropdown = widgets.Dropdown(
                options=[(f"{p.get('Name', '?')} (Id={p['Id']})", str(p["Id"])) for p in projects],
                description="Project:",
                layout=widgets.Layout(width="80%"),
                style={"description_width": "initial"},
            )
            save_button = widgets.Button(description="Use this project", button_style="primary")
            output = widgets.Output()
            _project_lookup = {str(p["Id"]): p.get("Name", "") for p in projects}

            def _on_select(_):
                with output:
                    output.clear_output()
                    picked_id = dropdown.value
                    picked_name = _project_lookup[picked_id]
                    _save_selection_to_endpoint(picked_id, picked_name)
                    print(f"Saved project selection: {picked_name} (Id={picked_id})")
                    print(f"Saved to {ENDPOINT_CONFIG_PATH}")
                    print(f"\nRe-run this cell to continue with file discovery.")

            save_button.on_click(_on_select)
            display(widgets.VBox([dropdown, save_button, output]))
            raise RuntimeError(
                "Pick a project from the dropdown above, click Use this project, "
                "then re-run this cell to continue."
            )

# BASESPACE_PROJECT_ID is now guaranteed set. Fetch files in the project.
print(f"\nListing files in BaseSpace project {BASESPACE_PROJECT_ID}...\n")
stdout = _run_bs(["bs", "contents", "project", "-i", BASESPACE_PROJECT_ID, "-f", "json"])
project_files = json.loads(stdout)
total_files_in_project = len(project_files)

# Categorize (for reporting) and filter.
fastq_files = [f for f in project_files if _is_fastq(f.get("Name", ""))]
non_fastq_files = [f for f in project_files if not _is_fastq(f.get("Name", ""))]

if SELECTED_FILE_IDS:
    # Explicit hand-pick overrides both the FASTQ filter and the "all files"
    # default. Users know exactly what they want.
    wanted = set(map(str, SELECTED_FILE_IDS))
    selected = [f for f in project_files if str(f.get("Id")) in wanted]
    if not selected:
        raise RuntimeError(
            "None of the SELECTED_FILE_IDS matched files in this project. "
            "Check the IDs (BaseSpace uses numeric IDs as strings)."
        )
    print(f"Filtered to {len(selected)} file(s) matching SELECTED_FILE_IDS "
          f"(FASTQ_ONLY filter ignored since SELECTED_FILE_IDS is set).")
elif FASTQ_ONLY:
    selected = fastq_files
    print(f"Project contains {total_files_in_project} files total: "
          f"{len(fastq_files)} FASTQ + {len(non_fastq_files)} auxiliary "
          f"(BaseSpace workflow metadata, log/config/db files).")
    print(f"FASTQ_ONLY=True, so transferring {len(selected)} FASTQ file(s) only.")
    print(f"  (set FASTQ_ONLY=False in the parameter cell to transfer everything.)")
else:
    selected = project_files
    print(f"FASTQ_ONLY=False, transferring ALL {len(selected)} files "
          f"(includes {len(non_fastq_files)} auxiliary files).")

# Report the transfer set.
total_size = sum(int(f.get("Size", 0)) for f in selected)
print(f"\nSelected for transfer:\n")
for f in selected[:20]:
    size_mb = int(f.get("Size", 0)) / 1024 / 1024
    print(f"  {f.get('Id', '?'):>12}  {size_mb:>8.1f} MB  {f.get('Name', '?')}")
if len(selected) > 20:
    print(f"  ... and {len(selected) - 20} more")
print(f"\nTotal: {total_size / 1024 / 1024 / 1024:.2f} GB")


## Pre-transfer summary

A summary of what's about to happen. If the numbers look wrong (unexpected file count, unexpectedly large size), stop and adjust `SELECTED_FILE_IDS` or `BASESPACE_PROJECT_ID` in the parameter cell before continuing.


In [ ]:
# Rough time estimate. Workbench-to-GCS throughput is typically ~500 MB/s
# aggregate, but BaseSpace's outbound is usually the bottleneck. Use
# ~100 MB/s as a conservative estimate; real-world runs usually beat this.
est_seconds = (total_size / 1024 / 1024) / 100  # bytes -> MB / (MB/s)
est_minutes = est_seconds / 60

print(f"About to transfer:")
print(f"  Source:      BaseSpace project {BASESPACE_PROJECT_ID}")
print(f"  Destination: {INGEST_BUCKET_URL}")
print(f"  Files:       {len(selected)}")
print(f"  Size:        {total_size / 1024 / 1024 / 1024:.2f} GB")
print(f"  Est. time:   ~{est_minutes:.0f} min at 100 MB/s (usually faster)")
print()
print(f"After files land in the ingest bucket, the APGAP backend cascade")
print(f"(DLP scan -> SRA scrubber -> move to lab bucket) takes over")
print(f"automatically. You'll see files disappear from the ingest bucket")
print(f"over the next few minutes as they get processed. That's expected.")


## Running the transfer

For each file:

1. Check the local transfer log at `~/.apgap/transferred.json` — skip if already transferred in a previous run
2. Check if the file is already in the ingest bucket — skip if so (mid-processing from an interrupted run)
3. Download from BaseSpace to `LOCAL_STAGING_DIR` on the Workbench
4. Upload to the ingest bucket using the endpoint's service account key
5. Record in the transfer log
6. Delete the local temp file to reclaim disk

Sequential by default — network is the bottleneck, not CPU, so parallelizing adds complexity for marginal gain on typical workloads. If a file fails partway through, safe to re-run this cell; already-transferred files are skipped via the log.

**Re-transferring after you've deleted files from the lab** (three options, ordered by granularity):

- **All files**: set `FORCE_REUPLOAD = True` in the parameter cell, or delete the whole log: `rm ~/.apgap/transferred.json`
- **A specific list**: set `SELECTED_FILE_IDS = ["<id1>", "<id2>"]` + `FORCE_REUPLOAD = True` in the parameter cell
- **Surgical**: edit `~/.apgap/transferred.json` and remove the specific filenames you want to re-transfer

Why the local log exists: the scrubber cascade moves files out of the ingest bucket into your lab's target bucket, so a naive "is it in the ingest bucket?" check would say "no, re-upload" on every run and create duplicates in your lab. The endpoint's narrow SA can't see the lab bucket directly, so we track what we've transferred locally.


In [ ]:
import json
import os
import subprocess
import time
from datetime import datetime, timezone

from google.cloud import storage
from google.oauth2 import service_account

# Load SA credentials and open a scoped GCS client.
sa_creds = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_KEY_PATH)
storage_client = storage.Client(credentials=sa_creds, project=sa_key["project_id"])

bucket_name = INGEST_BUCKET_URL.replace("gs://", "").rstrip("/")
bucket = storage_client.bucket(bucket_name)

# Sanity-check bucket access before spending time on downloads. Deliberately
# NOT using bucket.reload() here -- that calls storage.buckets.get, which is
# a bucket-level permission the endpoint's SA doesn't have (the SA is scoped
# to object-level permissions only: storage.objectViewer/Creator/User).
# Instead probe an object-level operation the SA does have: check if a fake
# blob "exists". Returns False for a non-existent object but the underlying
# API call verifies storage.objects.get is granted; a 403 would raise here
# instead of hitting us for every file in the transfer loop.
try:
    bucket.blob("__nb07_permission_check__").exists()
except Exception as e:
    raise RuntimeError(
        f"Could not access ingest bucket {INGEST_BUCKET_URL} with the "
        f"provided SA key: {e}\n"
        f"Common causes: endpoint expired (check TTL in Portal), wrong "
        f"bucket URL, or SA key belongs to a different endpoint. "
        f"Delete ~/.apgap/ and re-paste from a fresh Portal endpoint."
    )

# Local transfer log to prevent re-uploads across runs.
#
# The scrubber cascade moves files OUT of the ingest bucket into the lab's
# target bucket. So after a successful transfer, the ingest bucket goes
# empty and a naive `blob.exists()` check would say "not there, re-upload"
# on the next run. That would create duplicates in the lab bucket (the
# backend renames re-uploads with -1, -2 suffixes via generate_unique_filename).
#
# The endpoint's narrow SA can't see the lab's target bucket to check for
# duplicates over there. So we track transfers locally instead: after each
# successful upload we append the filename + BaseSpace file_id + timestamp
# + bucket to transferred.json. On subsequent runs, we skip files that
# appear in the log.
#
# Overriding the log:
#   - FORCE_REUPLOAD=True in the parameter cell: bypass the log entirely
#   - rm ~/.apgap/transferred.json: nuke the whole log
#   - Edit transferred.json manually: surgical removal of specific entries
TRANSFER_LOG_PATH = os.path.join(
    os.path.dirname(SERVICE_ACCOUNT_KEY_PATH), "transferred.json"
)


def _load_transfer_log():
    if not os.path.exists(TRANSFER_LOG_PATH):
        return {}
    try:
        return json.load(open(TRANSFER_LOG_PATH))
    except Exception:
        # Corrupted log -- start fresh rather than crash the whole run.
        return {}


def _log_transfer(filename, file_id, size, bucket_name):
    log = _load_transfer_log()
    log[filename] = {
        "file_id": file_id,
        "size": size,
        "bucket": bucket_name,
        "at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    with open(TRANSFER_LOG_PATH, "w") as f:
        json.dump(log, f, indent=2, sort_keys=True)


def _find_downloaded_file(staging_dir, filename):
    """After `bs download`, the file may land at `staging_dir/filename` (the
    common case) OR inside a subdirectory (some BaseSpace files like
    workflow logs get nested under their originating AppResult folder).
    Search recursively for `filename` and return the first match.
    """
    for root, _dirs, files in os.walk(staging_dir):
        if filename in files:
            return os.path.join(root, filename)
    return None


transferred_before = _load_transfer_log()

if FORCE_REUPLOAD:
    print(f"FORCE_REUPLOAD=True: transfer log will be ignored; all selected "
          f"files will be re-downloaded and re-uploaded.\n"
          f"  (log at {TRANSFER_LOG_PATH} will still be updated as files transfer.)\n")

total_bytes = 0
completed = 0
skipped_bucket = 0
skipped_log = 0
failed = []
transfer_start = time.time()

for i, f in enumerate(selected, 1):
    file_id = str(f.get("Id"))
    filename = f.get("Name", f"file-{file_id}")
    size_mb = int(f.get("Size", 0)) / 1024 / 1024

    print(f"[{i}/{len(selected)}] {filename} ({size_mb:.1f} MB)")
    file_start = time.time()

    # Skip if we've already transferred this file in a previous run (per
    # the local log). Prevents duplicate uploads to the lab after the
    # scrubber cascade emptied the ingest bucket. FORCE_REUPLOAD bypasses.
    if not FORCE_REUPLOAD and filename in transferred_before:
        prev = transferred_before[filename]
        print(f"  -> already transferred on {prev.get('at', '?')}, skipping")
        print(f"     (set FORCE_REUPLOAD=True to bypass, or rm {TRANSFER_LOG_PATH})")
        skipped_log += 1
        continue

    # Skip if still in the ingest bucket (mid-processing from a prior run
    # that was interrupted before the file moved through the scrubber).
    blob = bucket.blob(filename)
    if blob.exists():
        print(f"  -> already in ingest bucket, skipping")
        _log_transfer(filename, file_id, int(f.get("Size", 0)), bucket_name)
        skipped_bucket += 1
        continue

    # Download from BaseSpace to local staging. Print full stderr on
    # failure so users see the real error instead of a truncated snippet.
    dl_cmd = ["bs", "download", "file", "-i", file_id, "--output", LOCAL_STAGING_DIR]
    dl_result = subprocess.run(dl_cmd, capture_output=True, text=True)
    if dl_result.returncode != 0:
        print(f"  ! BaseSpace download failed (exit {dl_result.returncode}):")
        print(f"    command: {' '.join(dl_cmd)}")
        for line in (dl_result.stderr or "").splitlines():
            print(f"    {line}")
        failed.append((file_id, filename, f"download exit {dl_result.returncode}"))
        continue

    # bs may write the file to a subdirectory under LOCAL_STAGING_DIR
    # instead of the flat expected path -- recursive search handles both.
    local_path = _find_downloaded_file(LOCAL_STAGING_DIR, filename)
    if local_path is None:
        print(f"  ! Downloaded file not found anywhere under {LOCAL_STAGING_DIR}")
        print(f"    (bs download reported success but no file appeared -- possibly a bs bug)")
        failed.append((file_id, filename, "post-download missing"))
        continue

    # Upload to ingest bucket.
    try:
        blob.upload_from_filename(local_path)
    except Exception as e:
        print(f"  ! GCS upload failed:")
        print(f"    {type(e).__name__}: {e}")
        failed.append((file_id, filename, f"upload {type(e).__name__}"))
        # Leave the local file so a re-run can retry from the upload step
        # without re-downloading.
        continue

    # Record in transfer log so future runs skip this file.
    _log_transfer(filename, file_id, int(f.get("Size", 0)), bucket_name)

    # Cleanup local staging to reclaim disk.
    os.remove(local_path)

    elapsed = time.time() - file_start
    mbps = size_mb / elapsed if elapsed > 0 else 0
    total_bytes += int(f.get("Size", 0))
    completed += 1
    print(f"  ok {elapsed:.1f}s ({mbps:.1f} MB/s)")

total_elapsed = time.time() - transfer_start
print(f"\nTransfer summary:")
print(f"  Uploaded: {completed}")
print(f"  Skipped (transfer log):    {skipped_log}")
print(f"  Skipped (still in bucket): {skipped_bucket}")
print(f"  Failed:                    {len(failed)}")
print(f"  Wall clock: {total_elapsed:.0f}s")
print(f"  Bytes moved: {total_bytes / 1024 / 1024 / 1024:.2f} GB")
if total_elapsed > 0 and total_bytes > 0:
    print(f"  Avg throughput: {(total_bytes / 1024 / 1024) / total_elapsed:.1f} MB/s")

if failed:
    print(f"\nFailed files:")
    for file_id, filename, reason in failed:
        print(f"  {file_id}  {filename}: {reason}")
    print("\nRe-run this cell to retry. Already-uploaded files are skipped.")
    print("If failures repeat, see the 'Getting help' section at the bottom of the notebook.")


## What happens next

Files are now in the ingest bucket. From here the APGAP backend handles the rest:

1. **Pub/Sub notification** fires on each new object → backend gets pinged
2. **DLP scan** runs to detect PII (~1-3 min per file)
3. **SRA scrubber workflow** removes human reads (~5-15 min per file depending on size)
4. **Final move** — clean files land in your lab's target GCS bucket

Files disappear from the ingest bucket as they progress. Watch progress in the APGAP portal's Uploads / Files view.

If a scrub fails (unlikely on well-formed FASTQ), the file stays in a failed state and you get a Portal notification.


In [ ]:
# Quick inventory of what's currently in the ingest bucket. Depending on
# how fast the backend cascade runs, some files may already be gone.
# That's normal and means processing is in flight.
print(f"Current ingest bucket contents:\n")
found = 0
for blob in bucket.list_blobs():
    size_mb = (blob.size or 0) / 1024 / 1024
    print(f"  {size_mb:>8.1f} MB  {blob.name}")
    found += 1

if found == 0:
    print("  (empty; files may have already moved through the scrubber cascade)")

print(f"\n{found} file(s) still in ingest bucket.")
print("Once the cascade processes them, they show up in your lab bucket")
print("and are visible in nb 02.")


## Getting help

If a cell fails and the error message isn't clear, do this before opening an issue:

1. **Run the failing command in a JupyterLab terminal.** Every `subprocess.run` call in this notebook is a plain shell command; the failing cell output usually shows the exact `bs`/`gsutil` command that was invoked. Run it directly to see the un-truncated stderr — that's where the real error lives.
2. **Check the versions** that are installed:
   ```
   ~/.local/bin/bs --version
   python -c "import google.cloud.storage; print(google.cloud.storage.__version__)"
   ```
   These versions are pinned in the install cell; if yours differ, that's a good thing to include in the issue.
3. **File an issue** at [azpathogens/apgap-notebooks/issues](https://github.com/azpathogens/apgap-notebooks/issues) with:
   - The notebook number and cell number that failed
   - The full error output (screenshot or copy-paste)
   - What you were trying to do (which BaseSpace project, roughly how many files)
   - The version output from step 2

## Where to go next

- **See your data**: [02-read-your-data.ipynb](02-read-your-data.ipynb) — after the scrubber cascade completes, the copied files show up in your lab's analytical dataset bucket
- **Launch a pipeline**: [03-launch-a-pipeline.ipynb](03-launch-a-pipeline.ipynb) for viralrecon on the copied FASTQs
- **Prepare an NCBI submission**: [06-launch-tostadas.ipynb](06-launch-tostadas.ipynb)
- **Back to the** [getting-started overview](01-getting-started.ipynb)

External docs:

- [BaseSpace CLI reference](https://developer.basespace.illumina.com/docs/content/documentation/cli/cli-overview)
- Concepts (BaseSpace, DLP scanning, SRA scrubber, batch upload endpoints): [05-reference.ipynb](05-reference.ipynb)
